# NB11a - Base integrated models (2004-2017)

In [ ]:
import shap #SHAP package
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from lightgbm import LGBMClassifier
from PyALE import ale
from sklearn.inspection import partial_dependence
import warnings
warnings.filterwarnings("ignore")

In [ ]:
BASE = Path("..").resolve()
DATA = BASE / "data/engineered" / "parquet_saves"/ "me_merged_base.parquet"
RESULTS = BASE / "outputs/results" / "11a"
OUTPUT = BASE / "outputs/attribution" / "outputs_11a"
BEST_PARAMS = RESULTS / "best_params_log.pkl"

OUTPUT.mkdir(parents=True, exist_ok=True)

In [ ]:
df = pd.read_parquet(DATA)
with open(BEST_PARAMS, "rb") as f:
    best_params = pickle.load(f)

print(df.shape)
print(best_params.keys())

In [ ]:
print(df.columns.tolist())

## Replicate 11a

### Data handling

In [ ]:
id_cols = ['bank_id', 'country_iso', 'year']
country_col = 'country_iso'
target = 'crisis'
macro_cols = ['NGDP_RPCH','NGDPD', 'PCPIPCH', 'LUR', 'BCA_NGDPD', 'BIS_REER', 'RXF11FX_REVS', 'reer_g', 'fx_g']
micro_cols = [c for c in df.columns if c not in id_cols + [target] + macro_cols]
all_cols = micro_cols + macro_cols

print("Micro:", micro_cols)
print("Macro:", macro_cols)
print(f"{len(micro_cols)} |micro features, {len(macro_cols)} macro features, {len(all_cols)} total features")

In [ ]:
# Drop decisions per feature set - determined by Belsley CI + VDP + economic reasoning.
# When running on a new dataset, call find_collinear_pairs() and record decisions here.
DROPS = {
    'micro': [
        'x_assets', 'x_equity', 'x_liab', 'x_liab_equity',
        'x_loans_net',
        'exp_total',
    ],
    'macro': [],  # fill after inspecting macro_pairs output above
}

micro_cols_clean = [c for c in micro_cols if c not in DROPS['micro']]
macro_cols_clean = [c for c in macro_cols if c not in DROPS['macro']]
all_cols_clean   = [c for c in all_cols   if c not in DROPS['micro'] + DROPS['macro']]

print(f"Micro:      {len(micro_cols)} -> {len(micro_cols_clean)} variables after drops")
print(f"Macro:      {len(macro_cols)} -> {len(macro_cols_clean)} variables after drops")
print(f"Integrated: {len(all_cols)} -> {len(all_cols_clean)} variables after drops")
print(f"\nMicro clean: {micro_cols_clean}")
print(f"Macro clean: {macro_cols_clean}")

# SHAP

## Model dictionary for SHAP

In [ ]:
MODEL_MAP = {
    'logistic_regression': LogisticRegression,
    'random_forest': RandomForestClassifier,
    'mlp': MLPClassifier,
    'decision_tree': DecisionTreeClassifier,
    'xgboost': XGBClassifier,
    'lightgbm': LGBMClassifier,
}

feature_sets = {
    'micro': micro_cols_clean,
    'macro': macro_cols_clean,
    'integrated': all_cols_clean,
}

In [ ]:
shap_records = []
pd_records = []
ale_records = []

## No need to rerun!

In [ ]:
for variant, feature_set, test_year, model_name in best_params.keys():
    print(f"{variant} | {feature_set} | {test_year} | {model_name}")
    cols = feature_sets[feature_set]
    params = best_params[(variant, feature_set, test_year, model_name)]

    train_full = df[df['year'] < test_year]
    test = df[df['year'] == test_year]
    y_test = test[target]

    use_rt1 = (variant == 'rt1')
    if use_rt1:
        df_sorted   = train_full.sort_values([country_col, 'year'])
        prev_crisis = df_sorted.groupby(country_col)[target].shift(1).fillna(0)
        onset_mask  = (df_sorted[target] == 1) & (prev_crisis == 0)
        onset_rows  = df_sorted[onset_mask][[country_col, 'year']]
        exclude_set = (
            set(zip(onset_rows[country_col], onset_rows['year'])) |
            set(zip(onset_rows[country_col], onset_rows['year'] + 1))
        )
        train = train_full[~train_full.apply(
            lambda r: (r[country_col], r['year']) in exclude_set, axis=1
        )]
        if train[target].sum() == 0:
            print(f"  test={test_year} | RT1  no crisis obs after onset filter")
            continue
    else:
        train = train_full
        
    y_train = train[target]
    n_neg_ratio = (y_train ==0).sum() / max((y_train == 1).sum(), 1)
    
    imputer = SimpleImputer(strategy='median')
    X_train_imp = imputer.fit_transform(train[cols])
    X_test_imp = imputer.transform(test[cols])

    if model_name == 'logistic_regression':
        scaler        = StandardScaler()
        X_train_final = scaler.fit_transform(X_train_imp)
        X_test_final  = scaler.transform(X_test_imp)
        model = LogisticRegression(
            solver='lbfgs',
            C=params.get('C', 1.0),
            max_iter=1000, class_weight='balanced', random_state=42
        )

    elif model_name == 'random_forest':
        X_train_final = X_train_imp
        X_test_final  = X_test_imp
        model = RandomForestClassifier(
                n_estimators=params.get('n_estimators', 100),
                max_depth=params.get('max_depth', None),
                min_samples_leaf=params.get('min_samples_leaf', 1),
                max_features=params.get('max_features', 'sqrt'),
                class_weight='balanced', random_state=42
            )

    elif model_name == 'xgboost':
        X_train_final = X_train_imp
        X_test_final  = X_test_imp
        model = XGBClassifier(
            n_estimators=params.get('n_estimators', 100),
            max_depth=params.get('max_depth', 6),
            learning_rate=params.get('learning_rate', 0.1),
            subsample=params.get('subsample', 1.0),
            min_child_weight=params.get('min_child_weight', 1),
            scale_pos_weight=n_neg_ratio,
            eval_metric='logloss', random_state=42
        )

    elif model_name == 'mlp':
        scaler        = StandardScaler()
        X_train_final = scaler.fit_transform(X_train_imp)
        X_test_final  = scaler.transform(X_test_imp)
        arch = params.get('hidden_layer_sizes', '128')
        hidden_layer_sizes = {'32': (32,), '64': (64,), '128': (128,), '64_32': (64, 32), '128_64': (128, 64)}.get(arch, (64,))
        model = MLPClassifier(
            hidden_layer_sizes=hidden_layer_sizes,
            learning_rate_init=params.get('learning_rate_init', 1e-3),
            alpha=params.get('alpha', 1e-4),
            max_iter=5000,
            tol=1e-3,  # relaxed from default 1e-4 - suppresses warnings on flat loss landscapes without affecting AUROC
            random_state=42
        )

    elif model_name == 'decision_tree':
        X_train_final = X_train_imp
        X_test_final  = X_test_imp
        model = DecisionTreeClassifier(
            max_depth=params.get('max_depth', 5),
            min_samples_leaf=params.get('min_samples_leaf', 5),
            class_weight='balanced', random_state=42
        )

    elif model_name == 'lightgbm':
        # Wrap as DataFrame: ensures consistent feature names for fit and predict
        X_train_final = pd.DataFrame(X_train_imp, columns=cols)
        X_test_final  = pd.DataFrame(X_test_imp,  columns=cols)
        model = LGBMClassifier(
            n_estimators=params.get('n_estimators', 100),
            max_depth=params.get('max_depth', 6),
            learning_rate=params.get('learning_rate', 0.1),
            num_leaves=params.get('num_leaves', 31),
            subsample=params.get('subsample', 1.0),
            min_child_samples=params.get('min_child_samples', 20),
            scale_pos_weight=n_neg_ratio,
            random_state=42, verbose=-1
        )

    sw = np.where(y_train == 1, n_neg_ratio, 1.0) if model_name == 'mlp' else None
    if model_name == 'mlp':
        model.fit(X_train_final, y_train, sample_weight=sw)
    else:
        model.fit(X_train_final, y_train)
    
    # SHAP values computations
    if model_name in ['random_forest', 'xgboost', 'lightgbm', 'decision_tree']:
        explainer = shap.TreeExplainer(model)
        sv = explainer.shap_values(X_test_final)
        if isinstance(sv, list):
            shap_vals = sv[1]
        elif sv.ndim ==3:
            shap_vals = sv[:,:, 1]
        else:
            shap_vals = sv
    
    elif model_name == 'logistic_regression':
        explainer = shap.LinearExplainer(model, X_train_final)
        shap_vals = explainer.shap_values(X_test_final)

    elif model_name == 'mlp':
        X_background = shap.sample(X_train_final, 50)
        explainer = shap.KernelExplainer(model.predict_proba, X_background)
        sv = explainer.shap_values(X_test_final, nsamples=50)
        if isinstance(sv, list):
            shap_vals = sv[1]          # older SHAP: list of [class_0, class_1]
        elif sv.ndim == 3:
            shap_vals = sv[:, :, 1]    # newer SHAP: (n_samples, n_features, n_classes)
        else:
            shap_vals = sv             # already (n_samples, n_features)

    shap_df = pd.DataFrame(shap_vals, columns=[f'shap_{c}' for c in cols])
    shap_df['variant'] = variant
    shap_df['feature_set'] = feature_set
    shap_df['test_year'] = test_year
    shap_df['model_name'] = model_name
    shap_df['bank_id'] = test['bank_id'].values
    shap_df['year'] = test['year'].values

    shap_records.append(shap_df)
    
    del model
    del explainer
pd.concat(shap_records, ignore_index=True).to_parquet(OUTPUT / "shap_values_11a.parquet", index=False)
pd.concat(shap_records, ignore_index=True).to_csv(OUTPUT / "shap_values_11a.csv", index=False)

# ALE & PD

### Compute mean absolute SHAP per feature

In [ ]:
shap_values = pd.read_parquet(OUTPUT / "shap_values_11a.parquet")
shap_int = shap_values[shap_values['feature_set'] == 'integrated']

### Features found by SHAP

In [ ]:
shap_cols = [c for c in shap_int.columns if c.startswith('shap_')]
mean_abs_shap = shap_int[shap_cols].abs().mean().sort_values(ascending=False)

# KNOWN ISSUE (README, Known issues 5): micro/macro split by name prefix, so inc_nii, inc_op and cr_impchg count as macro. This feeds top5_micro and so the H-statistic and PD pairs below: in 11a inc_nii and inc_op (true micro top 5) are left out and p_nim, x_ib_liab take their place. Kept as run.
shap_micro_cols = [c for c in mean_abs_shap.index if c.startswith(('shap_g_','shap_l_','shap_r_','shap_c_','shap_x_','shap_p_'))]
shap_macro_cols = [c for c in mean_abs_shap.index if c not in shap_micro_cols]

Top features across all models, all test years and for both baseline and rt1, used for pair selection in PDP

In [ ]:
top5_micro = mean_abs_shap[shap_micro_cols].head(5)
top5_macro = mean_abs_shap[shap_macro_cols].head(5)

top5_micro = [c[5:] for c in top5_micro.index]
top5_macro = [c[5:] for c in top5_macro.index]
print(top5_macro)
print(top5_micro)

## Pairs for PD

In [ ]:
data_pairs = [(m, k) for m in top5_micro for k in top5_macro]

Theoretically driven pairs

In [ ]:
fixed_pairs = [
    ('c_eqratio', 'NGDP_RPCH'),
    ('c_leverage', 'LUR'),
    ('x_loans_gross', 'PCPIPCH'),
]

In [ ]:
all_pairs = list(set(fixed_pairs + data_pairs))
print(f"Total pairs: {len(all_pairs)}")

In [ ]:
integrated_keys = [(v, fs, ty, mn) for v, fs, ty, mn in best_params.keys() if fs == 'integrated']
len(integrated_keys)

## ALE and PD

In [ ]:
import logging
from tqdm import tqdm
logging.getLogger('PyALE._ALE_generic').setLevel(logging.WARNING)


class ProbaWrapper:
    """Expose predict_proba[:, 1] as .predict so PyALE reads the crisis PROBABILITY.

    PyALE calls model.predict(X) internally (PyALE/_src/ALE_1D.py). For a classifier that
    returns hard 0/1 labels, and in this rare-event setting almost nothing crosses the 0.5
    class boundary when a feature is perturbed, so the effect collapses to zero: the earlier
    ALE tables had random forest at exactly 0.000 for 20 of the 24 features. Passing
    probabilities is the documented PyALE pattern for classifiers.

    The scaler is applied here rather than to X, so the ALE grid can stay in the feature's
    real units while the model still receives the scaled input it was trained on.
    """

    def __init__(self, model, scaler=None, columns=None):
        self.model, self.scaler, self.columns = model, scaler, columns

    def predict(self, X):
        Xv = X[self.columns] if self.columns is not None else X
        if self.scaler is not None:
            Xv = self.scaler.transform(Xv)
        if isinstance(self.model, LGBMClassifier):
            # fitted on a named DataFrame - keep the names or LightGBM warns on mismatch
            Xv = pd.DataFrame(np.asarray(Xv), columns=self.columns)
        elif hasattr(Xv, "to_numpy"):
            # every other model was fitted on a plain array
            Xv = Xv.to_numpy()
        return self.model.predict_proba(Xv)[:, 1]


ale_records = []
pd_records = []

for variant, feature_set, test_year, model_name in tqdm(integrated_keys, desc='ALE/PD'):
    cols = feature_sets[feature_set]
    params = best_params[(variant, feature_set, test_year, model_name)]

    train_full = df[df['year'] < test_year]

    use_rt1 = (variant == 'rt1')
    if use_rt1:
        df_sorted   = train_full.sort_values([country_col, 'year'])
        prev_crisis = df_sorted.groupby(country_col)[target].shift(1).fillna(0)
        onset_mask  = (df_sorted[target] == 1) & (prev_crisis == 0)
        onset_rows  = df_sorted[onset_mask][[country_col, 'year']]
        exclude_set = (
            set(zip(onset_rows[country_col], onset_rows['year'])) |
            set(zip(onset_rows[country_col], onset_rows['year'] + 1))
        )
        train = train_full[~train_full.apply(
            lambda r: (r[country_col], r['year']) in exclude_set, axis=1
        )]
        if train[target].sum() == 0:
            print(f"  test={test_year} | RT1 SKIPPED no crisis obs after onset filter")
            continue
    else:
        train = train_full
        
    y_train = train[target]
    n_neg_ratio = (y_train ==0).sum() / max((y_train == 1).sum(), 1)
    
    scaler = None  # reset per iteration: only LR/MLP set one; must not leak to the next model
    imputer = SimpleImputer(strategy='median')
    X_train_imp = imputer.fit_transform(train[cols])

    if model_name == 'logistic_regression':
        scaler        = StandardScaler()
        X_train_final = scaler.fit_transform(X_train_imp)
        model = LogisticRegression(
            solver='lbfgs',
            C=params.get('C', 1.0),
            max_iter=1000, class_weight='balanced', random_state=42
        )

    elif model_name == 'random_forest':
        X_train_final = X_train_imp
        model = RandomForestClassifier(
                n_estimators=params.get('n_estimators', 100),
                max_depth=params.get('max_depth', None),
                min_samples_leaf=params.get('min_samples_leaf', 1),
                max_features=params.get('max_features', 'sqrt'),
                class_weight='balanced', random_state=42
            )

    elif model_name == 'xgboost':
        X_train_final = X_train_imp
        model = XGBClassifier(
            n_estimators=params.get('n_estimators', 100),
            max_depth=params.get('max_depth', 6),
            learning_rate=params.get('learning_rate', 0.1),
            subsample=params.get('subsample', 1.0),
            min_child_weight=params.get('min_child_weight', 1),
            scale_pos_weight=n_neg_ratio,
            eval_metric='logloss', random_state=42
        )

    elif model_name == 'mlp':
        scaler        = StandardScaler()
        X_train_final = scaler.fit_transform(X_train_imp)
        arch = params.get('hidden_layer_sizes', '128')
        hidden_layer_sizes = {'32': (32,), '64': (64,), '128': (128,), '64_32': (64, 32), '128_64': (128, 64)}.get(arch, (64,))
        model = MLPClassifier(
            hidden_layer_sizes=hidden_layer_sizes,
            learning_rate_init=params.get('learning_rate_init', 1e-3),
            alpha=params.get('alpha', 1e-4),
            max_iter=5000,
            tol=1e-3,  # relaxed from default 1e-4 - suppresses warnings on flat loss landscapes without affecting AUROC
            random_state=42
        )

    elif model_name == 'decision_tree':
        X_train_final = X_train_imp
        model = DecisionTreeClassifier(
            max_depth=params.get('max_depth', 5),
            min_samples_leaf=params.get('min_samples_leaf', 5),
            class_weight='balanced', random_state=42
        )

    elif model_name == 'lightgbm':
        # Wrap as DataFrame: ensures consistent feature names for fit and predict
        X_train_final = pd.DataFrame(X_train_imp, columns=cols)
        model = LGBMClassifier(
            n_estimators=params.get('n_estimators', 100),
            max_depth=params.get('max_depth', 6),
            learning_rate=params.get('learning_rate', 0.1),
            num_leaves=params.get('num_leaves', 31),
            subsample=params.get('subsample', 1.0),
            min_child_samples=params.get('min_child_samples', 20),
            scale_pos_weight=n_neg_ratio,
            random_state=42, verbose=-1
        )

    sw = np.where(y_train == 1, n_neg_ratio, 1.0) if model_name == 'mlp' else None
    if model_name == 'mlp':
        model.fit(X_train_final, y_train, sample_weight=sw)
    else:
        model.fit(X_train_final, y_train)
    
    X_train_df = pd.DataFrame(X_train_final, columns=cols)
    # Subsample to 300 rows for ALE/PD — sufficient for bin-based estimation, avoids O(n) cost on full panel
    X_pd = X_train_df.sample(min(300, len(X_train_df)), random_state=42).reset_index(drop=True)

    # ALE uses the SAME rows in RAW units, so the grid is in real feature units and is
    # comparable across models; the scaler is applied inside ProbaWrapper instead.
    X_pd_raw = (pd.DataFrame(X_train_imp, columns=cols)
                  .sample(min(300, len(X_train_imp)), random_state=42)
                  .reset_index(drop=True))
    ale_model = ProbaWrapper(model, scaler=scaler, columns=cols)

    for feat in all_cols_clean:
        ale_result = ale(X=X_pd_raw, model=ale_model, feature=[feat], grid_size=20, plot=False)
        ale_result = ale_result.reset_index()
        ale_result['variant'] = variant
        ale_result['feature_set'] = feature_set
        ale_result['test_year'] = test_year
        ale_result['model_name'] = model_name
        ale_records.append(ale_result)
    
    for feat1, feat2 in all_pairs:
        feat1_idx = cols.index(feat1)
        feat2_idx = cols.index(feat2)
        pd_result = partial_dependence(estimator=model, X=X_pd, features=[(feat1_idx, feat2_idx)], kind='average', grid_resolution=10)
        grid1 = pd_result['grid_values'][0]
        grid2 = pd_result['grid_values'][1]
        pd_vals = pd_result['average'][0]

        for i, v1 in enumerate(grid1):
            for j, v2 in enumerate(grid2):
                pd_records.append({
                    'feat1': feat1,
                    'feat2': feat2,
                    'feat1_val': v1,
                    'feat2_val': v2,
                    'pd_value': pd_vals[i,j],
                    'variant': variant,
                    'feature_set': feature_set,
                    'test_year': test_year,
                    'model_name': model_name
                })
    del model

pd.concat(ale_records, ignore_index=True).to_parquet(OUTPUT / "ale_values_11a.parquet", index=False)
pd.DataFrame(pd_records).to_parquet(OUTPUT / "pd_values_11a.parquet", index=False)

# SHAP analysis

## Global Feature Importance (Aggregated across all windows)

In [ ]:
shap_values = pd.read_parquet(OUTPUT / "shap_values_11a.parquet")
shap_int = shap_values[shap_values['feature_set'] == 'integrated']
shap_cols = [c for c in shap_int.columns if c.startswith('shap_')]

# KNOWN ISSUE (README, Known issues 5): micro/macro split by name prefix, so inc_nii, inc_op and cr_impchg count as macro here. Affects this cell's summaries and figures only; the parquets are written earlier.
micro_prefixes = ('g_', 'l_', 'r_', 'c_', 'x_', 'p_')
(OUTPUT / 'figures').mkdir(exist_ok=True)

# Aggregated across models, per variant
for variant_name in ['baseline_t1', 'rt1']:
    subset = shap_int[shap_int['variant'] == variant_name]
    mean_abs_shap = subset[shap_cols].abs().mean().sort_values(ascending=False)
    mean_abs_shap.index = [c[5:] for c in mean_abs_shap.index]
    colours = ['steelblue' if any(f.startswith(p) for p in micro_prefixes) else 'coral' for f in mean_abs_shap.index]
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(mean_abs_shap.index[::-1], mean_abs_shap.values[::-1], color=colours[::-1])
    ax.set_xlabel('Mean |SHAP value|')
    ax.set_title(f'Global Feature Importance - {variant_name} (NB11a)')
    plt.tight_layout()
    plt.savefig(OUTPUT / 'figures' / f'shap_global_importance_{variant_name}.png', dpi=150, bbox_inches='tight')
    plt.show()

# Per model per variant
for variant_name in ['baseline_t1', 'rt1']:
    for model_name in shap_int['model_name'].unique():
        subset = shap_int[(shap_int['variant'] == variant_name) & (shap_int['model_name'] == model_name)]
        mean_abs_shap = subset[shap_cols].abs().mean().sort_values(ascending=False)
        mean_abs_shap.index = [c[5:] for c in mean_abs_shap.index]
        colours = ['steelblue' if any(f.startswith(p) for p in micro_prefixes) else 'coral' for f in mean_abs_shap.index]
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.barh(mean_abs_shap.index[::-1], mean_abs_shap.values[::-1], color=colours[::-1])
        ax.set_xlabel('Mean |SHAP value|')
        ax.set_title(f'Global Feature Importance - {variant_name} - {model_name} (NB11a)')
        plt.tight_layout()
        plt.savefig(OUTPUT / 'figures' / f'shap_global_importance_{variant_name}_{model_name}.png', dpi=150, bbox_inches='tight')
        plt.show()

## Temporal Feature Importance (per variant per test year)

In [ ]:
shap_values = pd.read_parquet(OUTPUT / "shap_values_11a.parquet")
shap_int = shap_values[shap_values['feature_set'] == 'integrated']
shap_cols = [c for c in shap_int.columns if c.startswith('shap_')]
(OUTPUT / 'figures').mkdir(exist_ok=True)

# Aggregated across models, per variant
for variant_name in ['baseline_t1', 'rt1']:
    subset = shap_int[shap_int['variant'] == variant_name]
    temporal_shap = subset.groupby('test_year')[shap_cols].mean().abs()
    temporal_shap.columns = [c[5:] for c in temporal_shap.columns]
    fig, ax = plt.subplots(figsize=(12, 8))
    sns.heatmap(temporal_shap.T, cmap='YlOrRd', ax=ax)
    ax.set_title(f'Feature Importance per Test Year - {variant_name} (NB11a)')
    ax.set_xlabel('Test year')
    ax.set_ylabel('Feature')
    plt.tight_layout()
    plt.savefig(OUTPUT / 'figures' / f'shap_temporal_{variant_name}.png', dpi=150, bbox_inches='tight')
    plt.show()

# Per model per variant
for variant_name in ['baseline_t1', 'rt1']:
    for model_name in shap_int['model_name'].unique():
        subset = shap_int[(shap_int['variant'] == variant_name) & (shap_int['model_name'] == model_name)]
        temporal_shap = subset.groupby('test_year')[shap_cols].mean().abs()
        temporal_shap.columns = [c[5:] for c in temporal_shap.columns]
        fig, ax = plt.subplots(figsize=(12, 8))
        sns.heatmap(temporal_shap.T, cmap='YlOrRd', ax=ax)
        ax.set_title(f'Feature Importance per Test Year - {variant_name} - {model_name} (NB11a)')
        ax.set_xlabel('Test year')
        ax.set_ylabel('Feature')
        plt.tight_layout()
        plt.savefig(OUTPUT / 'figures' / f'shap_temporal_{variant_name}_{model_name}.png', dpi=150, bbox_inches='tight')
        plt.show()

## Local explanations (LATER)

## Micro vs macro contribution split (sum SHAP by prefix group)

In [ ]:
shap_values = pd.read_parquet(OUTPUT / "shap_values_11a.parquet")
shap_int = shap_values[shap_values['feature_set'] == 'integrated']
shap_cols = [c for c in shap_int.columns if c.startswith('shap_')]

# KNOWN ISSUE (README, Known issues 5): micro/macro split by name prefix, so inc_nii, inc_op and cr_impchg count as macro here. Affects this cell's summaries and figures only; the parquets are written earlier.
micro_prefixes = ('g_', 'l_', 'r_', 'c_', 'x_', 'p_')
shap_micro = [c for c in shap_cols if any(c[5:].startswith(p) for p in micro_prefixes)]
shap_macro = [c for c in shap_cols if c not in shap_micro]
(OUTPUT / 'figures').mkdir(exist_ok=True)

# Aggregated across models, per variant
for variant_name in ['baseline_t1', 'rt1']:
    subset = shap_int[shap_int['variant'] == variant_name]
    agg = pd.Series({'micro': subset[shap_micro].abs().mean().mean(), 'macro': subset[shap_macro].abs().mean().mean()})
    fig, ax = plt.subplots(figsize=(5, 4))
    agg.plot(kind='bar', color=['steelblue', 'coral'], ax=ax)
    ax.set_title(f'Micro vs Macro Contribution - {variant_name} (NB11a)')
    ax.set_ylabel('Mean |SHAP value|')
    ax.set_xticklabels(['Micro', 'Macro'], rotation=0)
    plt.tight_layout()
    plt.savefig(OUTPUT / 'figures' / f'shap_micro_macro_agg_{variant_name}.png', dpi=150, bbox_inches='tight')
    plt.show()

# Per model per variant
for variant_name in ['baseline_t1', 'rt1']:
    subset = shap_int[shap_int['variant'] == variant_name]
    split = subset.groupby('model_name').apply(lambda g: pd.Series({
        'micro': g[shap_micro].abs().mean().mean(),
        'macro': g[shap_macro].abs().mean().mean()
    }))
    split.plot(kind='bar', figsize=(10, 5), color=['steelblue', 'coral'])
    plt.title(f'Micro vs Macro SHAP Contribution by Model - {variant_name} (NB11a)')
    plt.xlabel('Model')
    plt.ylabel('Mean |SHAP value|')
    plt.xticks(rotation=45, ha='right')
    plt.legend(['Micro', 'Macro'])
    plt.tight_layout()
    plt.savefig(OUTPUT / 'figures' / f'shap_micro_macro_split_{variant_name}.png', dpi=150, bbox_inches='tight')
    plt.show()

# H-Statistics

H measures how much of the joint prediction variance cannot be explained by the two features acting independently. H close to 0 = no interaction, H close to 1 = strong interaction

In [ ]:
pd_values = pd.read_parquet(OUTPUT / "pd_values_11a.parquet")
interaction_records = []

for (variant, model_name, test_year, feat1, feat2), group in pd_values.groupby(['variant', 'model_name', 'test_year', 'feat1', 'feat2']):
    grid = group.pivot(index='feat1_val', columns='feat2_val', values='pd_value').values

    # Friedman & Popescu's H^2 is defined on CENTRED partial-dependence functions:
    #   H2 = sum[ PDjk - PDj - PDk ]^2 / sum[ PDjk ]^2 , every term mean-zero.
    # sklearn's partial_dependence returns the average PREDICTION (a probability sitting
    # around the base rate mu), not a deviation from it. Without subtracting mu the identity
    # becomes (mu + fj + fk + I) - (mu + fj) - (mu + fk) = I - mu, so the statistic measures
    # the base rate rather than the interaction. Under the uncentred version a completely
    # FLAT surface scored H = 1.000 - the maximum - and the result correlated -0.87 with the
    # PD range, i.e. it was reading flatness as interaction. Hence the explicit centring below.
    mu = grid.mean()
    pd1 = np.mean(grid, axis=1, keepdims=True)   # PD_j, marginalised over feat2
    pd2 = np.mean(grid, axis=0, keepdims=True)   # PD_k, marginalised over feat1

    numerator = np.sum((grid - pd1 - pd2 + mu) ** 2)   # + mu restores the once-too-often subtraction
    denominator = np.sum((grid - mu) ** 2)             # total variation of the joint surface

    # A surface with no variation at all carries no interaction, but would divide 0 by ~0 and
    # return noise. Report it as missing rather than letting a degenerate ratio into the figures.
    h2 = numerator / denominator if denominator > 1e-12 else np.nan

    # Keep the old uncentred value alongside it so the correction stays auditable and the
    # difference between the two can be quoted without recomputing anything.
    h2_uncentred = np.sum((grid - pd1 - pd2) ** 2) / np.sum(grid ** 2)

    interaction_records.append({
        'variant': variant,
        'test_year': test_year,
        'model_name': model_name,
        'feat1': feat1,
        'feat2': feat2,
        'h2': h2,
        'h2_uncentred': h2_uncentred,
        'pd_range': grid.max() - grid.min(),   # diagnostic: flat surfaces are what broke the old version
    })

h_df = pd.DataFrame(interaction_records)
h_df.to_parquet(OUTPUT / "h_statistic_11a.parquet", index=False)

print(f"pairs: {len(h_df)} | mean H (centred): {h_df['h2'].mean():.4f} | "
      f"mean H (old, uncentred): {h_df['h2_uncentred'].mean():.4f} | "
      f"share >= 0.10: {(h_df['h2'] >= 0.10).mean():.3f}")

# Average H across test years
h_mean = h_df.groupby(['variant', 'model_name', 'feat1', 'feat2'])['h2'].mean().reset_index()
(OUTPUT / 'figures').mkdir(exist_ok=True)

# The colour scale is pinned to 0-1, the theoretical range of H. Letting seaborn autoscale
# would stretch differences of a few thousandths across the full palette and make near-zero
# interaction look substantial - the visual version of the bug being fixed here.
HEAT = dict(cmap='YlOrRd', annot=True, fmt='.3f', vmin=0.0, vmax=1.0)

# Aggregated across models, per variant
h_agg_all = h_mean.groupby(['variant', 'feat1', 'feat2'])['h2'].mean().reset_index()
for variant_name in ['baseline_t1', 'rt1']:
    h_pivot = h_agg_all[h_agg_all['variant'] == variant_name].pivot(index='feat1', columns='feat2', values='h2')
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(h_pivot, ax=ax, **HEAT)
    ax.set_title(f'H-statistic (Micro x Macro Interactions) - {variant_name} (NB11a)')
    plt.tight_layout()
    plt.savefig(OUTPUT / 'figures' / f'h_statistic_{variant_name}.png', dpi=150, bbox_inches='tight')
    plt.show()

# Per model per variant
for variant_name in ['baseline_t1', 'rt1']:
    for model_name in h_mean['model_name'].unique():
        h_agg = h_mean[(h_mean['variant'] == variant_name) & (h_mean['model_name'] == model_name)]
        h_pivot = h_agg.pivot(index='feat1', columns='feat2', values='h2')
        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(h_pivot, ax=ax, **HEAT)
        ax.set_title(f'H-statistic - {variant_name} - {model_name} (NB11a)')
        plt.tight_layout()
        plt.savefig(OUTPUT / 'figures' / f'h_statistic_{variant_name}_{model_name}.png', dpi=150, bbox_inches='tight')
        plt.show()

# ALE Plots

In [ ]:
# Run this after loading to verify PyALE columns before adjusting meta_cols in cell below
# print(pd.read_parquet(OUTPUT / 'ale_values_11a.parquet').columns.tolist())

In [ ]:
# Derive top features from SHAP parquet (standalone - does not depend on Section 4 variables)
shap_values = pd.read_parquet(OUTPUT / "shap_values_11a.parquet")
shap_int_ale = shap_values[shap_values['feature_set'] == 'integrated']
shap_cols_ale = [c for c in shap_int_ale.columns if c.startswith('shap_')]
mean_abs_shap_ale = shap_int_ale[shap_cols_ale].abs().mean().sort_values(ascending=False)
mean_abs_shap_ale.index = [c[5:] for c in mean_abs_shap_ale.index]
# KNOWN ISSUE (README, Known issues 5): micro/macro split by name prefix, so inc_nii, inc_op and cr_impchg count as macro here. Affects this cell's summaries and figures only; the parquets are written earlier.
micro_prefixes = ('g_', 'l_', 'r_', 'c_', 'x_', 'p_')
_micro = [c for c in mean_abs_shap_ale.index if any(c.startswith(p) for p in micro_prefixes)]
_macro = [c for c in mean_abs_shap_ale.index if c not in _micro]
# Top 10 overall — let rankings determine the mix, not a forced 5+5 split
selected_features = list(mean_abs_shap_ale.head(10).index)

# Load ALE parquet and normalise to long form
ale_raw = pd.read_parquet(OUTPUT / "ale_values_11a.parquet")
(OUTPUT / 'figures').mkdir(exist_ok=True)

# Adjust meta_cols after inspecting ale_raw.columns if PyALE added extra columns (e.g. size, CIs)
meta_cols = ['variant', 'feature_set', 'test_year', 'model_name', 'eff']
feature_cols = [c for c in ale_raw.columns if c not in meta_cols]

ale_df = ale_raw.melt(
    id_vars=meta_cols,
    value_vars=feature_cols,
    var_name='feature',
    value_name='feature_val'
).dropna(subset=['feature_val'])

In [ ]:
(OUTPUT / 'figures').mkdir(exist_ok=True)

# Aggregated across variants, all models as lines
for feat in selected_features:
    subset = ale_df[ale_df['feature'] == feat]
    avg = subset.groupby(['model_name', 'feature_val'])['eff'].mean().reset_index()

    fig, ax = plt.subplots(figsize=(8, 4))
    for model_name in avg['model_name'].unique():
        m = avg[avg['model_name'] == model_name]
        ax.plot(m['feature_val'], m['eff'], label=model_name)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_xlabel(feat)
    ax.set_ylabel('ALE effect')
    ax.set_title(f'ALE - {feat} (NB11a)')
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT / 'figures' / f'ale_{feat}.png', dpi=150, bbox_inches='tight')
    plt.show()

# Per variant, all models as lines
for variant_name in ['baseline_t1', 'rt1']:
    for feat in selected_features:
        subset = ale_df[(ale_df['feature'] == feat) & (ale_df['variant'] == variant_name)]
        avg = subset.groupby(['model_name', 'feature_val'])['eff'].mean().reset_index()

        fig, ax = plt.subplots(figsize=(8, 4))
        for model_name in avg['model_name'].unique():
            m = avg[avg['model_name'] == model_name]
            ax.plot(m['feature_val'], m['eff'], label=model_name)
        ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
        ax.set_xlabel(feat)
        ax.set_ylabel('ALE effect')
        ax.set_title(f'ALE - {feat} - {variant_name} (NB11a)')
        ax.legend()
        plt.tight_layout()
        plt.savefig(OUTPUT / 'figures' / f'ale_{feat}_{variant_name}.png', dpi=150, bbox_inches='tight')
        plt.show()